## Analyze RAD/PACT Violations

The following code matches the list of RAD-converted addresses we made in 1_extract_developments.ipynb to [housing code violations](https://data.cityofnewyork.us/Housing-Development/Housing-Maintenance-Code-Violations/wvxf-dwi5/about_data), which are made public by HPD for all privately managed buildings in the city. 
<br>
<br>
This analysis merges HPD housing code violations with RAD developments by joining on the address column. Addresses come fully intact in the list of RAD developments. The housenumber and streetname columns in the HPD dataset were combined to form a full street address. 
<br>
<br>
This is a conservative approach to joining these two datasets together. The housing authority did it's own analysis and found 15,993 violations between 1/1/21 and 9/27/25 -- by merging on unique combinations of the BIN and BBL columns -- whereas the analysis below finds 14,281 during roughly the same time period.
<br>
<br>
In New York City, street addresses may sometimes be marked with two addresses, even if referring to the same exact location. NYCHA's analysis likely accounts for those extra violations attached to rarely used address assignments. 

In [1]:
## import libraries
import pandas as pd
import numpy as np
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
plt.style.use('ggplot')
from datetime import datetime, timedelta

In [2]:
## read in csv that includes developments
rad_buildings = pd.read_csv('../input/coded_files/addresses_updated_120525.csv', dtype={'bbl': 'object',
                                                                                        'bldg': 'object',
                                                                                        'zip code': 'object',
                                                                                        'cd': 'object',
                                                                                        'fc': 'object',
                                                                                        'ss': 'object',
                                                                                        'sa': 'object',
                                                                                        'cc': 'object',
                                                                                        'bin': 'object'})

In [3]:
rad_buildings.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1531 entries, 0 to 1530
Data columns (total 16 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   bbl          1508 non-null   object
 1   bldg         1431 non-null   object
 2   m            1127 non-null   object
 3   og_address   1530 non-null   object
 4   zip code     1527 non-null   object
 5   cd           1527 non-null   object
 6   fc           1527 non-null   object
 7   ss           1527 non-null   object
 8   sa           1527 non-null   object
 9   cc           1527 non-null   object
 10  bin          1508 non-null   object
 11  dev_name     1531 non-null   object
 12  boro         1531 non-null   object
 13  units        1514 non-null   object
 14  transfer     1510 non-null   object
 15  new_address  1530 non-null   object
dtypes: object(16)
memory usage: 191.5+ KB


In [4]:
print(f'unique BINs:{rad_buildings.bin.nunique()}\nunique addresses:{rad_buildings.new_address.nunique()}\nunique developments:{rad_buildings.dev_name.nunique()}')

unique BINs:714
unique addresses:1525
unique developments:101


In [ ]:
## read in violations
violations = pd.read_csv('../input/Housing_Maintenance_Code_Violations_20251123.csv', dtype = {'LowHouseNumber':'object',
                                                                                               'BIN':'object',
                                                                                               'HighHouseNumber':'object',
                                                                                               'HouseNumber':'object',
                                                                                               'ViolationID':'object',
                                                                                               'BuildingID':'object',
                                                                                               'BoroID':'object'})

In [ ]:
violations.shape

## Make a few changes

In [ ]:
## make the column names lowercase
violations.columns = violations.columns.str.lower()

In [ ]:
## combine house number with street name so that we can match with PACT development csv
violations['address'] = violations['housenumber'] + ' ' + violations['streetname']

In [ ]:
## get rid of columns we don't want/need
filtered_df = violations.loc[:,['violationid', 'buildingid', 'registrationid', 'boroid', 'borough',
                                'bin','bbl','housenumber', 'lowhousenumber', 'highhousenumber', 'streetname','class',
                                'inspectiondate', 'approveddate', 'originalcertifybydate',
                                'originalcorrectbydate', 'newcertifybydate', 'newcorrectbydate',
                                'certifieddate', 'ordernumber', 'novid', 'novdescription','novissueddate',
                                'currentstatusid', 'currentstatus','currentstatusdate', 'novtype', 
                                'violationstatus', 'rentimpairing','address']]

In [ ]:
## changing boro names to lowercase so we can match
boro_corrections = {'BROOKLYN':'brooklyn',
                    'MANHATTAN':'manhattan',
                    'QUEENS':'queens',
                    'BRONX':'bronx',
                    'STATEN ISLAND':'staten island'}

In [ ]:
## apply the corrections
filtered_df['borough'] = filtered_df['borough'].replace(boro_corrections)

In [ ]:
## renaming boro column in rad df
rad_buildings = rad_buildings.rename(columns = {'boro':'borough',
                                                'new_address':'address'})

## Merge

- left: use only keys from left frame, similar to a SQL left outer join; preserve key order.
- right: use only keys from right frame, similar to a SQL right outer join; preserve key order.
- outer: use union of keys from both frames, similar to a SQL full outer join; sort keys lexicographically.
- inner: use intersection of keys from both frames, similar to a SQL inner join; preserve the order of the left keys.
- cross: creates the cartesian product from both frames, preserves the order of the left keys.

In [ ]:
## stripping of whitespace just in case, so that i can merge
rad_buildings['borough'] = rad_buildings['borough'].str.strip()
filtered_df['borough'] = filtered_df['borough'].str.strip()

In [ ]:
## merging the two dfs on address bc it's the most unique measure we have for the buildings
## BBLs and BINs, for example, can be the same for different addresses
combined_df = pd.merge(filtered_df,
                       rad_buildings,
                       on = ['address','borough'],
                       how = 'left',
                       indicator=True)

In [ ]:
## using the indicator column to filter for addresses that exist in both dfs
both_df = combined_df[combined_df['_merge'] == 'both']

In [ ]:
both_df.shape

## Clean

In [ ]:
## it looks like some addresses carried over even though there's no data for them in the actual HPD violations dataset, 
# so filtering those out based on the violationid.. which i assume all valid complaints should have...
violations_df = both_df.dropna(subset = 'violationid').reset_index(drop = True)

In [ ]:
violations_df.shape

In [ ]:
## change NOV issue date to datetime dtype
violations_df['novissueddate'] = violations_df['novissueddate'].astype('datetime64[ns]')
violations_df['inspectiondate'] = violations_df['inspectiondate'].astype('datetime64[ns]')
violations_df['approveddate'] = violations_df['approveddate'].astype('datetime64[ns]')
violations['originalcertifybydate'] = violations['originalcertifybydate'].astype('datetime64[ns]')
violations['originalcorrectbydate'] = violations['originalcorrectbydate'].astype('datetime64[ns]')

In [ ]:
## create year columns for inspections and notifications
violations_df['inspectiondate_year'] = violations_df['inspectiondate'].dt.year
violations_df['novissued_year'] = violations_df['novissueddate'].dt.year

In [ ]:
## inspect duplicated violations
unique_violations = violations_df[violations_df.duplicated(subset=['violationid'], keep=False)]
unique_violations.to_csv('../output/unique_violations_test.csv')

In [ ]:
## it looks like some are duplicates, while some have the same violation id but are for violations against different HPD codes (novid)
violations_df = violations_df.drop_duplicates(subset = ['violationid','novid'], keep = "first").reset_index(drop = True)

In [ ]:
violations_df.shape

In [ ]:
## write to a csv file
violations_df.to_csv('../output/violations_w_developments.csv', index = False)

## Violations between January 1, 2021 and September 25, 2025

In [ ]:
violations_df['inspectiondate'] = violations_df['inspectiondate'].astype('datetime64[ns]')

In [ ]:
five_year_df = violations_df[(violations_df['inspectiondate'] >= '2021-01-01') & (violations_df['inspectiondate'] <= '2025-09-25')].reset_index()

In [ ]:
five_year_df.head()

In [ ]:
five_year_df.shape

In [ ]:
five_year_df.address.nunique()

In [ ]:
five_year_df.dev_name.nunique()

In [ ]:
## how many units across these 85 developments? 
devs_and_units = five_year_df.groupby(['dev_name','units']).size().reset_index()
devs_and_units.to_csv('../output/devs_and_units.csv')

In [ ]:
five_year_df.columns

## How many violations do the developments rack up 1,2,3 years post-conversion?

In [ ]:
## creating a new date column indicating when the data was pulled
## violation status updated daily
five_year_df['date_pulled'] = '2025-09-25'

In [ ]:
## changing dtypes
five_year_df['date_pulled'] = five_year_df['date_pulled'].astype('datetime64[ns]')
five_year_df['currentstatusdate'] = five_year_df['currentstatusdate'].astype('datetime64[ns]')
five_year_df['transfer'] = five_year_df['transfer'].astype('datetime64[ns]')
five_year_df['inspectiondate'] = five_year_df['inspectiondate'].astype('datetime64[ns]')

In [ ]:
## how many days between the date in which this data was pulled and the transfer date?
five_year_df['days_converted'] = five_year_df['date_pulled'] - five_year_df['transfer'] 

## how many days between the transfer date and the date in which the violation was inspected?
five_year_df['vio_days_after_conversion'] = five_year_df['inspectiondate'] - five_year_df['transfer']

In [ ]:
## create a function to assign categories based on the number of days
def assign_post_conversion_year(days):
    if days <= timedelta(days =365):
        return '0'
    elif days <= timedelta(days =730):
        return '1'
    elif days <= timedelta(days =1095):
        return '2'
    elif days <= timedelta(days =1460):
        return '3'
    elif days <= timedelta(days =1825):
        return '4'
    elif days <= timedelta(days =2190):
        return '5'
    elif days <= timedelta(days =2555):
        return '6'
    elif days <= timedelta(days =2920):
        return '7'
    elif days <= timedelta(days =3285):
        return '8'
    else:
        return '9+ years'

In [ ]:
## apply the function to the df
five_year_df['vio_post_conv_yr'] = five_year_df['vio_days_after_conversion'].apply(assign_post_conversion_year)

## How long do violations stay open?

In [ ]:
## create a column for the number of days violations stay open
five_year_df['days_vio_open'] = np.where(
    five_year_df['violationstatus'] == 'Close', # <-- if violation status is closed
    (five_year_df['currentstatusdate'] - five_year_df['novissueddate']), # <-- then subtract the notice of vio date from the current status date
    (five_year_df['date_pulled'] - five_year_df['novissueddate']) # <-- else subtract notice of vio date from the date data was pulled
)

In [ ]:
def assign_num_years_open(days):
    if days <= timedelta(days =7):
        return 'a week or less'
    elif days <= timedelta(days=30):
        return 'one week to a month'
    elif days <= timedelta(days =90):
        return 'one to three months'
    elif days <= timedelta(days =180):
        return 'three to six months'
    elif days <= timedelta(days =364):
        return 'six months to a year'
    else:
        return 'a year or longer'

In [ ]:
## create categories/ranges for the number of days violations stay open
five_year_df['vio_open_range'] = five_year_df['days_vio_open'].apply(assign_num_years_open)

In [ ]:
## how long do ALL violations stay open?
five_year_df['vio_open_range'].value_counts().reset_index()

In [ ]:
## share of the whole
3691/14281

In [ ]:
## how long do violations remain open at each development?
violation_length = five_year_df.groupby(['dev_name','vio_open_range']).size().reset_index(name='open_range_vios')


In [ ]:
## filter for only more than a year
duration_df = violation_length[violation_length['vio_open_range']=='a year or longer']

In [ ]:
## make a df that has the total number of violations
tot_violations = five_year_df.dev_name.value_counts().reset_index(name='tot_vios')
tot_violations

In [ ]:
## merge
duration_w_tot = pd.merge(duration_df,
                          tot_violations,
                          on='dev_name',
                          how='left')

In [ ]:
## make a new column with the total open for more than a year as a percent of total violations
duration_w_tot['pct_open_range'] = duration_w_tot['open_range_vios']/duration_w_tot['tot_vios']

In [ ]:
duration_w_tot.sort_values(by='pct_open_range',ascending=False).head(18)

In [ ]:
duration_w_tot.dev_name.nunique()

## C-Class Violations

In [ ]:
## how long do Class-C violations, in particular, stay open?
violations_class_five_year = five_year_df[five_year_df['class'] == 'C']

In [ ]:
violations_class_five_year.dev_name.nunique()

In [ ]:
violations_class_five_year.address.nunique()

In [ ]:
414/564

In [ ]:
five_year_df.to_csv('../output/violations_21_25.csv')

## How are Ocean Bay, Linden, Boulevard doing?

In [ ]:
# ocean_bay = post_transfer_df[post_transfer_df['development'] == 'OCEAN BAY (BAYSIDE)']
# ocean_bay.to_csv('../output/ocean_bay.csv')

In [ ]:
# ocean_bay.transfer_date.unique()

In [ ]:
# ocean_bay.yr_post_conversion.value_counts()

In [ ]:
# linden_df = post_transfer_df[post_transfer_df['development'] == 'LINDEN']

In [ ]:
# linden_df.yr_post_conversion.value_counts()

In [ ]:
# linden_df.groupby(['transfer_date','yr_post_conversion'])['vio_open_range'].value_counts().reset_index().head()

In [ ]:
# boulevard_df = five_year_df[five_year_df['development'] == 'BOULEVARD']
# #boulevard_df.to_csv('../output/boulevard.csv')

In [ ]:
# boulevard_df.bin.nunique()

In [ ]:
# boulevard_df.full_address.nunique()

In [ ]:
# boulevard_df.shape

In [ ]:
# boulevard_df.yr_post_conversion.value_counts()

In [ ]:
# boulevard_df.groupby(['transfer_date','yr_post_conversion'])['vio_open_range'].value_counts().reset_index().head()